In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Telco.csv")

In [3]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [5]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [6]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7043 non-null   int64  
 1   MonthlyCharges  7043 non-null   float64
 2   TotalCharges    7032 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 165.2 KB


In [7]:
df["tenure_bucket"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, float("inf")],
    labels=["0-12mo", "13-24mo", "25mo+"]
)

In [8]:
df[["tenure", "tenure_bucket"]].head(10)

,tenure,tenure_bucket
0,1,0-12mo
1,34,25mo+
2,2,0-12mo
3,45,25mo+
4,2,0-12mo
5,8,0-12mo
6,22,13-24mo
7,10,0-12mo
8,28,25mo+
9,62,25mo+


In [9]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

In [10]:
df["total_services"] = (
    df[service_columns] == "Yes"
).sum(axis=1)

In [11]:
df["charges_per_tenure"] = np.where(
    df["tenure"] == 0,
    0,
    df["TotalCharges"] / df["tenure"]
)

In [12]:
df[["tenure", "TotalCharges", "charges_per_tenure"]].head(10)

,tenure,TotalCharges,charges_per_tenure
0,1,29.85,29.850000
1,34,1889.50,55.573529
2,2,108.15,54.075000
3,45,1840.75,40.905556
4,2,151.65,75.825000
5,8,820.50,102.562500
6,22,1949.40,88.609091
7,10,301.90,30.190000
8,28,3046.05,108.787500
9,62,3487.95,56.257258


In [18]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,total_services,charges_per_tenure
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12mo,1,29.850000
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.50,No,25mo+,2,55.573529
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12mo,2,54.075000
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25mo+,3,40.905556
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12mo,0,75.825000


In [19]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "total_services",
    "charges_per_tenure"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "tenure_bucket"
]

In [20]:
for col in categorical_features:
    print(col, ":", df[col].nunique(), "unique values")

gender : 2 unique values
Partner : 2 unique values
Dependents : 2 unique values
PhoneService : 2 unique values
MultipleLines : 3 unique values
InternetService : 3 unique values
OnlineSecurity : 3 unique values
OnlineBackup : 3 unique values
DeviceProtection : 3 unique values
TechSupport : 3 unique values
StreamingTV : 3 unique values
StreamingMovies : 3 unique values
Contract : 3 unique values
PaperlessBilling : 2 unique values
PaymentMethod : 4 unique values
tenure_bucket : 3 unique values


In [21]:
from sklearn.preprocessing import OneHotEncoder

onehot = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [23]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", scaler, numeric_features),
        ("cat", onehot, categorical_features)
    ]
)

In [24]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

In [25]:
X = X.drop("customerID", axis=1)

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [28]:
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (5634, 49)
Testing shape: (1409, 49)


In [29]:
!pip install category_encoders

In [30]:
import category_encoders as ce

In [31]:
X_train, X_test, y_train, y_test

(      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
 3738    Male              0      No         No      35           No   
 3151    Male              0     Yes        Yes      15          Yes   
 4860    Male              0     Yes        Yes      13           No   
 3867  Female              0     Yes         No      26          Yes   
 3810    Male              0     Yes        Yes       1          Yes   
 ...      ...            ...     ...        ...     ...          ...   
 6303  Female              0     Yes         No      71          Yes   
 6227    Male              0      No         No       2          Yes   
 4673  Female              1      No         No      25          Yes   
 2710  Female              0     Yes         No      24          Yes   
 5639    Male              0      No         No       6          Yes   
 
          MultipleLines InternetService       OnlineSecurity  \
 3738  No phone service             DSL                   No   
 3151 

In [32]:
from category_encoders import TargetEncoder

target_encoder = TargetEncoder(
    cols=["PaymentMethod"]
)

X_train_te = target_encoder.fit_transform(
    X_train[["PaymentMethod"]],
    y_train
)

X_test_te = target_encoder.transform(
    X_test[["PaymentMethod"]]
)

print(X_train_te.head())
print(X_test_te.head())

      PaymentMethod
3738       0.457430
3151       0.192846
4860       0.192846
3867       0.149217
3810       0.457430
      PaymentMethod
437        0.149217
2280       0.149217
2235       0.149217
4460       0.457430
3761       0.149217


In [38]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [39]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [43]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [45]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'total_services',
                                                   'charges_per_tenure']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod',
                                                   'tenure_bucket'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [46]:
y_pred = model_pipeline.predict(X_test)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.801277501774308


In [47]:
#task3


In [48]:
# Select numerical columns
corr_data = df[numeric_features].corr()

print(corr_data)

                      tenure  MonthlyCharges  TotalCharges  total_services  \
tenure              1.000000        0.247900      0.825880        0.494263   
MonthlyCharges      0.247900        1.000000      0.651065        0.724706   
TotalCharges        0.825880        0.651065      1.000000        0.746101   
total_services      0.494263        0.724706      0.746101        1.000000   
charges_per_tenure  0.249391        0.994355      0.650915        0.719781   

                    charges_per_tenure  
tenure                        0.249391  
MonthlyCharges                0.994355  
TotalCharges                  0.650915  
total_services                0.719781  
charges_per_tenure            1.000000  


In [51]:

corr_matrix = df[numeric_features].corr()

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        
        correlation = corr_matrix.iloc[i, j]
        
        if abs(correlation) > 0.9:
            print(
                corr_matrix.columns[i],
                "and",
                corr_matrix.columns[j],
                "=>",
                correlation
            )

MonthlyCharges and charges_per_tenure => 0.9943551749187854


In [52]:
from sklearn.feature_selection import SelectKBest, f_classif

X_numeric = df[numeric_features].copy()
y = df["Churn"].map({"Yes": 1, "No": 0})

In [53]:
X_numeric = X_numeric.fillna(X_numeric.median())

In [54]:
selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

X_selected = selector.fit_transform(
    X_numeric,
    y
)

In [55]:
scores = pd.DataFrame({
    "Feature": numeric_features,
    "F_score": selector.scores_,
    "p_value": selector.pvalues_
})

scores = scores.sort_values(
    "F_score",
    ascending=False
)

print(scores)

              Feature     F_score        p_value
0              tenure  997.268010  7.999058e-205
2        TotalCharges  290.439831   7.508609e-64
1      MonthlyCharges  273.463704   2.706646e-60
4  charges_per_tenure  273.299934   2.929469e-60
3      total_services   54.571769   1.671374e-13


In [56]:
#above p-value tells us whether the relationship is statistically significant.


In [57]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_numeric,
    y,
    random_state=42
)

In [58]:
mi_results = pd.DataFrame({
    "Feature": numeric_features,
    "Mutual_Information": mi_scores
})

mi_results = mi_results.sort_values(
    "Mutual_Information",
    ascending=False
)

print(mi_results)

              Feature  Mutual_Information
0              tenure            0.070693
1      MonthlyCharges            0.045006
2        TotalCharges            0.044457
3      total_services            0.037309
4  charges_per_tenure            0.035661


In [59]:
comparison = scores[["Feature", "F_score"]].merge(
    mi_results,
    on="Feature"
)

comparison["F_rank"] = comparison["F_score"].rank(
    ascending=False
)

comparison["MI_rank"] = comparison["Mutual_Information"].rank(
    ascending=False
)

comparison = comparison.sort_values("F_rank")

print(comparison)

              Feature     F_score  Mutual_Information  F_rank  MI_rank
0              tenure  997.268010            0.070693     1.0      1.0
1        TotalCharges  290.439831            0.044457     2.0      3.0
2      MonthlyCharges  273.463704            0.045006     3.0      2.0
3  charges_per_tenure  273.299934            0.035661     4.0      5.0
4      total_services   54.571769            0.037309     5.0      4.0


In [60]:
corr_matrix = df[numeric_features].corr()

print(corr_matrix)

                      tenure  MonthlyCharges  TotalCharges  total_services  \
tenure              1.000000        0.247900      0.825880        0.494263   
MonthlyCharges      0.247900        1.000000      0.651065        0.724706   
TotalCharges        0.825880        0.651065      1.000000        0.746101   
total_services      0.494263        0.724706      0.746101        1.000000   
charges_per_tenure  0.249391        0.994355      0.650915        0.719781   

                    charges_per_tenure  
tenure                        0.249391  
MonthlyCharges                0.994355  
TotalCharges                  0.650915  
total_services                0.719781  
charges_per_tenure            1.000000  


In [61]:
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        
        correlation = corr_matrix.iloc[i, j]
        
        if abs(correlation) > 0.9:
            print(
                corr_matrix.columns[i],
                "and",
                corr_matrix.columns[j],
                "=>",
                correlation
            )

MonthlyCharges and charges_per_tenure => 0.9943551749187854


In [62]:
#MonthlyCharges and charges_per_tenure had a correlation of 0.9944, 
#which is greater than the 0.9 threshold. 
#Therefore, they were considered highly redundant. 
#charges_per_tenure was excluded from the final numerical feature set while retaining the original feature MonthlyCharges


In [69]:
#following are the final results

In [63]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "total_services"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "tenure_bucket"
]

In [64]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [65]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [66]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges',
                                                   'total_services']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod',
                                                   'tenure_bucket'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [68]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.801277501774308

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409

